# Part 2: Diabetes

In this part of the assignment, you will build a predictive model for diabetes disease progression in the next year based on current observed features of disease symptoms. 

**Learning objectives.** You will:
1. Train and test a linear model using ordinary least squares regression.
2. Use numerical Python (NumPy) and the standard `sklearn` API in Python  
3. Train and test a quadratic model, comparing to the linear model and demonstrating overfitting
4. Apply regularization, specifically LASSO, to build a sparse linear model

The following code will download and preview three examples of the data. The ten features are as follows (in order):

- age age in years
- sex
- bmi body mass index
- bp average blood pressure
- s1 tc, total serum cholesterol
- s2 ldl, low-density lipoproteins
- s3 hdl, high-density lipoproteins
- s4 tch, total cholesterol / HDL
- s5 ltg, log of serum triglycerides level
- s6 glu, blood sugar level

The target value is a quantiative measure of disease progression after 1 year, where larger numbers are worse.

The code stores the feature matrix `X` as a two-dimensional NumPy array where each row corresponds to a data point and each column is a feature. The target value is stored as a one-dimensional NumPy array `y` where the index `i` element of `y` correpsonds to the row `i` data point of `X`.

Your overall goal in this part is to build and evaluate a linear model to predict the target variable `y` as a function of the ten features in `X`, and to identify which features are more significant for predicting `y`.

In [1]:
# Run but DO NOT MODIFY this code

from sklearn.datasets import load_diabetes

# Load the diabetes dataset
diabetes = load_diabetes(scaled = False)
print(diabetes.feature_names)

# Get the feature data and target variable
X = diabetes.data
y = diabetes.target

# Preview the first 3 data points
print(X[:3])
print(y[:3])

['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
[[ 59.       2.      32.1    101.     157.      93.2     38.       4.
    4.8598  87.    ]
 [ 48.       1.      21.6     87.     183.     103.2     70.       3.
    3.8918  69.    ]
 [ 72.       2.      30.5     93.     156.      93.6     41.       4.
    4.6728  85.    ]]
[151.  75. 141.]


## Task 1

Use `sklearn` to randomly split the input data into a [train and test partition](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), with 30% of the data reserved for testing. Use a random seed of `2025` for reproducibility of the results.

Print the number of data points in the resulting train and test partitions.

In [2]:
# Write task 1 code here
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2025)

print("Number of training points = ", len(X_train))
print("Number of test points = ", len(X_test))

Number of training points =  309
Number of test points =  133


## Task 2

Build a baseline prediction by computing the [average](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) target value of the training data and predicting this for average for every test data point.

For example, if the training data target values were `[2, 2, 5]` then you would compute the average as `3`. If there there were only two test data points, then your predictions would be simply `[3, 3]`.

Evaluate the [root mean squared error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html#sklearn.metrics.root_mean_squared_error) between the baseline and the test data.

This RMSE will serve as a benchmark - any useful model should achieve a lower error than this simple baseline.

In [3]:
# Write task 2 code here
import numpy as np
baseline_value = np.mean(y_train)
baseline_arr = np.full(len(y_test), baseline_value)

from sklearn.metrics import root_mean_squared_error

baseline_rmse = root_mean_squared_error(baseline_arr, y_test)
print("Baseline root mean squared error = ", baseline_rmse)

Baseline root mean squared error =  75.8165287907097


## Task 3

Use [`sklearn`](https://scikit-learn.org/stable/) to fit a linear predictive model on the training data using [ordinary least squares regression](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares). 

Evaluate the [root mean squared error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html#sklearn.metrics.root_mean_squared_error) of the model on **both** the training data **and** the test data (that is, the training error and the generalization error). Report both results.

Note that the model predictions on the test data may not be perfect, but they should improve meaningfully over the simple baseline from Task 2 or something is wrong.

In [4]:
# Write task 3 code here
# Train model
from sklearn import linear_model

all_data_reg = linear_model.LinearRegression()
all_data_reg.fit(X_train, y_train)

# evaluate model
from sklearn.metrics import root_mean_squared_error
# using training data
y_train_predicted_from_all_data = all_data_reg.predict(X_train)
rmse_train_all_data = root_mean_squared_error(y_train_predicted_from_all_data, y_train)
print("RMSE on the training data using all all features = ", rmse_train_all_data)
                                                       
# using test data
y_test_predicted_from_all_data = all_data_reg.predict(X_test)
rmse_test_all_data = root_mean_squared_error(y_test_predicted_from_all_data, y_test)
print("RMSE on the test data using all features = ", rmse_test_all_data)


RMSE on the training data using all all features =  52.97771955290354
RMSE on the test data using all features =  55.14187488833095


## Task 4

To understand which input features are most important for predicting diabetes progression, we need a model that can automatically select relevant features. The ordinary least squares model from Task 3 uses all features, making it harder to identify which ones truly matter.

Build a new linear model using [Lasso regression](https://scikit-learn.org/stable/modules/linear_model.html#lasso) that meets two criteria:
  - Performance: At most 10% greater error than the linear model with all the features in task 3. 
  - Sparsity: At least three model coefficients set to 0 (meaning the model does not use these features to make predictions). You can treat any coefficient less than 0.0001 as effectively 0 for this task.

You may need to try multiple vaues of the `alpha` *hyperparameter* to find a satisfy both constraints. For example, you can try [0.1, 1, 5, 10, ...]. The final LASSO model is only required to satisfy the above two criteria. Nevertheless, you should only evaluate error on the test dataset **once** after searching for such a value of `alpha`. Use [cross validation](https://scikit-learn.org/stable/modules/cross_validation.html) on the training data or split the training data into train and validation sets.

For your final Lasso model with the chosen `alpha` fit on all of the training data, report the [root mean squared error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html#sklearn.metrics.root_mean_squared_error) of the model predictions on the test data. Report the three or more features for which the model coefficients were set to 0 (see feature names/interpretations above). Also please explain why you selected this `alpha`.

In [7]:
# Write task 4 code here
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import numpy as np

# Split the training data further into train and validation sets
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=2025
)

# Benchmark OLS on validation set
ols_val = linear_model.LinearRegression()
ols_val.fit(X_train_sub, y_train_sub)
ols_val_pred = ols_val.predict(X_val)
ols_val_rmse = root_mean_squared_error(y_val, ols_val_pred)

print("OLS validation RMSE =", ols_val_rmse)

# Try a range of alpha values and see which gives the best validation RMSE
alphas = [10, 15, 20, 25, 30, 35, 40, 45, 50]
best_alpha = None

for alpha in alphas:
    model = linear_model.Lasso(alpha=alpha)
    model.fit(X_train_sub, y_train_sub)

    y_pred_val = model.predict(X_val)
    rmse_val = root_mean_squared_error(y_pred_val, y_val)
    zero_coef_count = np.sum(np.abs(model.coef_) < 0.0001)

    print(f"Using alpha = {alpha}, we get rmse = {rmse_val} and {np.sum(model.coef_ == 0)} coefficient(s) equal to 0")

    if (
    (rmse_val - ols_val_rmse) / ols_val_rmse <= 0.1
    ) and zero_coef_count >= 3:
        best_alpha = alpha
        break

if best_alpha == None:
    print("no qualifying alpha was found. Using alpha = 10")
    best_alpha = 10

# Pick the alpha with the lowest validation RMSE
print("Best alpha found using validation set:", best_alpha)

lasso_reg = linear_model.Lasso(alpha=best_alpha, random_state=2025)
lasso_reg.fit(X_train, y_train)

# On training data
y_pred_train_lasso = lasso_reg.predict(X_train)
rmse_train_lasso = root_mean_squared_error(y_pred_train_lasso, y_train)

print("We have coefficients of \n", lasso_reg.coef_)
print("Using the lasso model on the training data, we get an RMSE = ", rmse_train_lasso)

# on testing data
y_pred_test_lasso = lasso_reg.predict(X_test)
rmse_test_lasso = root_mean_squared_error(y_pred_test_lasso, y_test)

percent_change = ((rmse_test_lasso - rmse_test_all_data) / rmse_test_all_data) * 100

print("Using the lasso model on the test data, we get an RMSE = ", rmse_test_lasso)
print(f"The lasso model is {percent_change:.4}% worse than the old model")

zero_mask = np.abs(lasso_reg.coef_) < 0.0001
zero_features = np.array(diabetes.feature_names)[zero_mask]
print(f"Number of zeroed features: {len(zero_features)}")
print("Zeroed features:", list(zero_features))



OLS validation RMSE = 52.21834353191712
Using alpha = 10, we get rmse = 56.08927968769621 and 3 coefficient(s) equal to 0
Best alpha found using validation set: 10
We have coefficients of 
 [-0.          0.          6.57970635  1.0437622   1.05195482 -1.16825995
 -1.82276072  0.          0.          0.22676226]
Using the lasso model on the training data, we get an RMSE =  55.53804929816267
Using the lasso model on the test data, we get an RMSE =  56.30140397885992
The lasso model is 2.103% worse than the old model
Number of zeroed features: 4
Zeroed features: [np.str_('age'), np.str_('sex'), np.str_('s4'), np.str_('s5')]


*Report features and explain for task 4 here*

Using an `alpha` of 10, we are able to get an RMSE of ~56.3 on the test data. This reflects an increase in RMSE of approxiamtely ~2.1%. This is well below the required benchmark of an increase of 10%. This alpha was selected after splitting the training data into training and validation sets. We train models on increasing values of alpha and quit once we have found a value of alpha that hits our performance requirement and has at least three coefficients set to 0. Many values of alpha hit this benchmark, but 10 was the first one to satisfy the other criterium, that **≥3** coefficients must be set to 0. Using `alpha = 5`, we are able to get **3** coefficients to be set to 0. These coefficients correspond to
- age
- sex
- s4 tch, total cholesterol / HDL
- s5 ltg, log of serum triglycerides level

Due to `alpha = 10` satisfying the required criteria, it was selected as our value of alpha.

It is worth noting that after finding the smallest alpha that hits our performance requirement, we do not retrain the model with the entire training dataset. It was found that doing this retains four coefficients set to zero when using `alpha = 10` (`age`, `sex`, `s4`, and `s5`). `alpha = 10`is the first value of alpha in our samples that will satisfy the performance benchmark while setting at least three zero coefficients when evaluated on the validation set and refit on the entire dataset. Another constant that we use here is the size of our validation set. We set aside only 20% of the training data to be the validation set. Following the explicit task requirement to report the final model fit on all training data, we select and evaluate `alpha = 10.`